In [1]:
import geopy
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time
import datetime

import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql.functions import size, mean, explode, udf, count_distinct, abs, median
from pyspark.sql.types import IntegerType, ArrayType, FloatType
from pyspark.sql import functions as F
from geopy import distance

In [6]:
# Initiate spark session
spark = SparkSession.builder.getOrCreate()

# Load data
path = '../data/endomondoHR_proper.json'
df = spark.read.json(path)

# Function to get distance
@udf
def get_distance(lat_long):
    lat, long = lat_long
    total_distance = 0
    for i in range(1, len(lat)):
        start_point = lat[i - 1], long[i - 1]
        end_point = lat[i], long[i]
        difference = distance.geodesic(start_point, end_point).miles
        total_distance += difference

    return total_distance

@udf
def elapsed_end(arr):
    """
    Return time that the workout has elapsed since the beginning to end.
    """
    start = datetime.datetime.fromtimestamp(arr[0])
    end = datetime.datetime.fromtimestamp(arr[-1])

    return (end - start).seconds

# Calculate mean of array -- udf
array_mean = udf(lambda x: float(np.mean(x)), FloatType())

# Replace lat and long with distance
df = df.withColumn('lat_long', F.array(F.col('latitude'), F.col('longitude')))
df = df.withColumn('distance', get_distance('lat_long'))
df = df.withColumn('altitude', array_mean('altitude')) 
df = df.withColumn('heart_rate', array_mean('heart_rate'))
df = df.withColumn('duration', elapsed_end('timestamp'))


# Cast to double type for descriptive statistics
df = df.select(df.distance.cast('double'),
               df.duration.cast('double'),
               'gender', 'heart_rate', 'sport', 'userId', 'id')
# Drop unncessary columns
df = df.drop('lat_long')
df = df.drop('latitude')
df = df.drop('longitude')
df = df.drop('altitude')
df = df.drop('url')
df = df.drop('speed')
df = df.drop('timestamp')

In [7]:
df.printSchema()

root
 |-- distance: double (nullable = true)
 |-- duration: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- heart_rate: float (nullable = true)
 |-- sport: string (nullable = true)
 |-- userId: long (nullable = true)
 |-- id: long (nullable = true)



In [9]:
# Load metadata
metadata_path = '../data/endomondoMeta.json'
metadata = spark.read.json(metadata_path)

# Cast to double type for descriptive statistics
usable_metadata = metadata.select(metadata.calories.cast('double'),
                                  'id')


# Join main frame and metadata by the id of the rows
joined_df = df.join(usable_metadata, 'id')

# Negative calories fix
joined_df = joined_df.withColumn('calories', abs(joined_df.calories))
joined_df.select('calories').describe().show()

# Filter out unreasonable data points
temp_df = joined_df.filter(joined_df.duration < 18000) \
.filter(joined_df.calories != 0) \
.filter(joined_df.calories < 4000)


final_df = temp_df
final_df = final_df.drop('id')

+-------+-----------------+
|summary|         calories|
+-------+-----------------+
|  count|           167285|
|   mean|14061.03123736885|
| stddev|  5251500.2760311|
|    min|              0.0|
|    max|     2.14748006E9|
+-------+-----------------+



In [11]:
final_df.printSchema()

root
 |-- distance: double (nullable = true)
 |-- duration: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- heart_rate: float (nullable = true)
 |-- sport: string (nullable = true)
 |-- userId: long (nullable = true)
 |-- calories: double (nullable = true)



In [12]:
# Count user again
final_df.select('userId').distinct().count()

1056

In [13]:
# Write to csv file for modeling
final_df.repartition(1).write.csv("../data/main_derived.csv", header=True, mode='overwrite')